## Desenvolvido por: Lukas Oliveira Antunes
## Linkedin: https://www.linkedin.com/in/lukas-oliveira-4a0506166/
#  Clipe Musical com IA — Fase 1
## Análise Musical + Storyboard Automático
###  100% Gratuito · Upload dinâmico de música e imagens

| Tarefa | Ferramenta | Custo |
|---|---|---|
| Análise musical | Librosa | Grátis |
| Separação vocal | Demucs (Meta) | Grátis |
| Análise de imagens | Groq API — Llama 4 Scout Vision | Grátis |
| Geração do storyboard | Groq API — LLaMA 3.3 70b | Grátis |

**Como obter a Groq API Key (sem cartão):**
1. Acesse https://console.groq.com
2. Crie conta com Google ou GitHub
3. Menu lateral → **API Keys → Create API Key**

---
 **Antes de começar:** `Ambiente de execução → Alterar tipo → GPU T4`

##  Célula 1 — Instalação

In [ ]:
print('Instalando dependências...')
!pip install -q librosa soundfile groq
!pip install -q demucs
!pip install -q matplotlib numpy scipy
print('\n Tudo instalado!')

##  Célula 2 — Groq API Key

In [ ]:
from google.colab import userdata
import os

# OPÇÃO A: Secrets do Colab (recomendado)
# Painel esquerdo → ícone de cadeado → Adicionar secret
# Nome: GROQ_API_KEY | Valor: sua_chave
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print(' API Key carregada via Secrets!')
except:
    GROQ_API_KEY = 'cole-sua-chave-aqui'  # OPÇÃO B
    print('  API Key definida diretamente.')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

from groq import Groq
client = Groq(api_key=GROQ_API_KEY)
test = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': 'Responda apenas: OK'}],
    max_tokens=5
)
print(f' Groq conectado! Teste: {test.choices[0].message.content}')

## Célula 3 — Upload dinâmico da música e imagens

> **Aqui você escolhe qual música e quais imagens usar.**  
> Pode rodar esta célula novamente a qualquer momento para trocar tudo e refazer o clipe.
>
> - Música: qualquer `.mp3`
> - Imagens: **mínimo 4**, formatos `.jpg`, `.jpeg` ou `.png`

In [ ]:
from google.colab import files
import os, shutil, mimetypes
from IPython.display import display, Image as IPImage
import ipywidgets as widgets

# ── Pastas do projeto ──────────────────────────────────────────
BASE_DIR    = '/content/projeto'
MUSIC_DIR   = f'{BASE_DIR}/musica'
IMG_DIR     = f'{BASE_DIR}/imagens'
OUTPUT_DIR  = f'{BASE_DIR}/output'

# Limpa uploads anteriores para não misturar projetos
for d in [MUSIC_DIR, IMG_DIR, OUTPUT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

# ── Upload da música ───────────────────────────────────────────
print('═' * 55)
print('  PASSO 1 — Upload da música (.mp3)')
print('═' * 55)
uploaded_music = files.upload()

# Valida e salva
MUSIC_PATH = None
for filename, content in uploaded_music.items():
    if not filename.lower().endswith('.mp3'):
        print(f'  "{filename}" ignorado — apenas .mp3 é aceito.')
        continue
    dest = os.path.join(MUSIC_DIR, filename)
    with open(dest, 'wb') as f:
        f.write(content)
    MUSIC_PATH = dest
    size_mb = len(content) / (1024 * 1024)
    print(f' Música: {filename}  ({size_mb:.1f} MB)')

if MUSIC_PATH is None:
    raise ValueError(' Nenhum .mp3 válido enviado. Rode a célula novamente.')

# ── Upload das imagens ─────────────────────────────────────────
print()
print('═' * 55)
print('  PASSO 2 — Upload das imagens (mínimo 4)')
print('  Formatos aceitos: .jpg  .jpeg  .png')
print('═' * 55)
uploaded_images = files.upload()

IMAGE_PATHS = []
VALID_IMG_EXT = {'.jpg', '.jpeg', '.png'}

for filename, content in uploaded_images.items():
    ext = os.path.splitext(filename)[1].lower()
    if ext not in VALID_IMG_EXT:
        print(f'⚠️  "{filename}" ignorado — extensão não suportada.')
        continue
    dest = os.path.join(IMG_DIR, filename)
    with open(dest, 'wb') as f:
        f.write(content)
    IMAGE_PATHS.append(dest)
    size_kb = len(content) / 1024
    print(f'  ✓ {filename}  ({size_kb:.0f} KB)')

# Validação mínimo 4 imagens
if len(IMAGE_PATHS) < 4:
    raise ValueError(
        f' Você enviou {len(IMAGE_PATHS)} imagem(ns). '
        f'O mínimo são 4. Rode a célula novamente e selecione mais imagens.'
    )

# ── Resumo e preview ──────────────────────────────────────────
print()
print('═' * 55)
print(f'   Música  : {os.path.basename(MUSIC_PATH)}')
print(f'   Imagens : {len(IMAGE_PATHS)} arquivo(s)')
print('═' * 55)

# Preview das imagens em grid
print('\nPreview das imagens enviadas:')
for path in IMAGE_PATHS:
    print(f'  → {os.path.basename(path)}')
    display(IPImage(filename=path, width=200))

##  Célula 4 — Análise musical com Librosa

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import json

print(f' Analisando: {os.path.basename(MUSIC_PATH)}')
y, sr = librosa.load(MUSIC_PATH, sr=None)
duracao_total = librosa.get_duration(y=y, sr=sr)
print(f'   Duração: {duracao_total:.2f}s ({duracao_total/60:.1f} min) | Sample rate: {sr} Hz')

# BPM e batidas
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
beat_times = librosa.frames_to_time(beat_frames, sr=sr)
bpm = float(tempo)
print(f'\n BPM: {bpm:.1f} | Batidas detectadas: {len(beat_times)}')

# Energia RMS
hop_length = 512
rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
rms_times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop_length)

# Segmentação estrutural
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=hop_length)
bounds = librosa.segment.agglomerative(mfcc, k=8)
bound_times = librosa.frames_to_time(bounds, sr=sr, hop_length=hop_length)

secoes = []
for i in range(len(bound_times) - 1):
    inicio = float(bound_times[i])
    fim    = float(bound_times[i + 1])
    mask   = (rms_times >= inicio) & (rms_times < fim)
    energia = float(np.mean(rms[mask])) if mask.any() else 0.0
    batidas_secao = int(np.sum((beat_times >= inicio) & (beat_times < fim)))
    secoes.append({
        'id': i + 1,
        'inicio': round(inicio, 2),
        'fim':    round(fim, 2),
        'duracao': round(fim - inicio, 2),
        'energia_normalizada': round(float(energia / rms.max()), 3),
        'batidas': batidas_secao
    })

energias = [s['energia_normalizada'] for s in secoes]
p33, p66 = np.percentile(energias, 33), np.percentile(energias, 66)
for s in secoes:
    e = s['energia_normalizada']
    s['intensidade'] = 'baixa' if e < p33 else ('alta' if e >= p66 else 'média')

print(f'\n {len(secoes)} seções detectadas:')
for s in secoes:
    print(f"   #{s['id']:2d}: {s['inicio']:6.1f}s → {s['fim']:6.1f}s | {s['intensidade']:5s} | {s['batidas']} batidas")

# ── Gráfico ────────────────────────────────────────────────────
nome_musica = os.path.splitext(os.path.basename(MUSIC_PATH))[0]
output_grafico = os.path.join(OUTPUT_DIR, f'{nome_musica}_analise.png')

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
fig.suptitle(f'Análise Musical — {nome_musica}', fontsize=14)

librosa.display.waveshow(y, sr=sr, ax=axes[0], color='steelblue', alpha=0.6)
cores = {'baixa': '#90CAF9', 'média': '#FFB74D', 'alta': '#EF5350'}
for s in secoes:
    axes[0].axvspan(s['inicio'], s['fim'], color=cores[s['intensidade']], alpha=0.25)
axes[0].set_title('Waveform + Seções  (azul=baixa · laranja=média · vermelho=alta)')
axes[0].set_xlabel('Tempo (s)')

axes[1].plot(rms_times, rms, color='darkorange', linewidth=0.8)
for bt in beat_times[::4]:
    axes[1].axvline(bt, color='gray', alpha=0.3, linewidth=0.5)
axes[1].set_title(f'Energia RMS | BPM: {bpm:.1f}')
axes[1].set_xlabel('Tempo (s)')

plt.tight_layout()
plt.savefig(output_grafico, dpi=150, bbox_inches='tight')
plt.show()

# ── Salva JSON ─────────────────────────────────────────────────
output_json_musical = os.path.join(OUTPUT_DIR, f'{nome_musica}_analise.json')
analise_musical = {
    'nome_musica':    nome_musica,
    'arquivo':        MUSIC_PATH,
    'duracao_total':  round(duracao_total, 2),
    'bpm':            round(bpm, 1),
    'sample_rate':    sr,
    'total_batidas':  len(beat_times),
    'secoes':         secoes
}
with open(output_json_musical, 'w', encoding='utf-8') as f:
    json.dump(analise_musical, f, ensure_ascii=False, indent=2)

print(f'\n JSON salvo: {output_json_musical}')
print(f' Gráfico salvo: {output_grafico}')

##  Célula 5 — Separação vocal com Demucs
 *~3–5 min. O vocal isolado será usado no lip sync na Fase 3.*

In [ ]:
import subprocess

demucs_out = os.path.join(OUTPUT_DIR, 'demucs')
os.makedirs(demucs_out, exist_ok=True)

print(f' Separando vocal de: {os.path.basename(MUSIC_PATH)}')
print('   Modelo: htdemucs | Modo: vocals vs no_vocals')
print('   Aguarde...\n')

cmd = [
    'python', '-m', 'demucs',
    '--two-stems', 'vocals',
    '-n', 'htdemucs',
    '-o', demucs_out,
    MUSIC_PATH
]
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(' Separação concluída!')
else:
    print(' Log Demucs (últimas linhas):')
    print(result.stderr[-1500:])

# Localiza os arquivos gerados dinamicamente
VOCAL_PATH        = None
INSTRUMENTAL_PATH = None

print('\n Arquivos gerados:')
for root, dirs, fls in os.walk(demucs_out):
    for fn in fls:
        full = os.path.join(root, fn)
        size = os.path.getsize(full) / (1024 * 1024)
        print(f'   {fn}  ({size:.1f} MB)')
        fn_lower = fn.lower()
        if 'vocals' in fn_lower and 'no_' not in fn_lower:
            VOCAL_PATH = full
        elif 'no_vocals' in fn_lower:
            INSTRUMENTAL_PATH = full

print(f'\n🎤 Vocal isolado     → {VOCAL_PATH}')
print(f'🎸 Instrumental      → {INSTRUMENTAL_PATH}')

# Atualiza o JSON com os caminhos
analise_musical['vocal_path']        = VOCAL_PATH
analise_musical['instrumental_path'] = INSTRUMENTAL_PATH
with open(output_json_musical, 'w', encoding='utf-8') as f:
    json.dump(analise_musical, f, ensure_ascii=False, indent=2)

print('\n✅ JSON atualizado com caminhos do Demucs.')

##  Célula 6 — Análise de imagens com Llama 4 Scout Vision (Groq)

In [ ]:
import base64, mimetypes, time, json
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def encode_image(path):
    mime, _ = mimetypes.guess_type(path)
    mime = mime or 'image/jpeg'
    with open(path, 'rb') as f:
        data = base64.standard_b64encode(f.read()).decode('utf-8')
    return mime, data

def analisar_imagem(path, idx, total):
    mime, data = encode_image(path)
    print(f'   [{idx}/{total}] Analisando: {os.path.basename(path)}...')
    resp = client.chat.completions.create(
        model='meta-llama/llama-4-scout-17b-16e-instruct',
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:{mime};base64,{data}'}},
            {'type': 'text', 'text': """Describe this image in detail for a music video AI pipeline. Cover:
- Person: appearance, face, hair, clothing, skin tone, approximate age, visual style
- Environment: location type, lighting, colors, mood/atmosphere
- Photo style: cinematic style, camera angle, composition
- 3 music video scene ideas that match this visual style
- A detailed Stable Diffusion prompt in English to recreate this exact look"""}
        ]}],
        max_tokens=800
    )
    return resp.choices[0].message.content

total = len(IMAGE_PATHS)
VISION_MODEL = 'meta-llama/llama-4-scout-17b-16e-instruct'
print(f'  Analisando {total} imagens com Llama 4 Scout Vision (Groq)...\n')
print(f'   Modelo: {VISION_MODEL}\n')

descricoes = []
for i, path in enumerate(IMAGE_PATHS):
    desc = analisar_imagem(path, i + 1, total)
    descricoes.append({'imagem': os.path.basename(path), 'descricao': desc})
    if i < total - 1:
        time.sleep(1)  # pequena pausa entre requisições

print(f'\n {total} imagens analisadas! Consolidando perfil...')

todas = '\n\n'.join(
    [f"Image {i+1} ({d['imagem']}):\n{d['descricao']}" for i, d in enumerate(descricoes)]
)

consolidacao = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': f"""Based on the descriptions of {total} images below, create a consolidated visual profile for AI music video production.

{todas}

Return ONLY a valid JSON (no extra text, no ```json``` markers):
{{
  "personagem": {{
    "descricao_geral": "", "rosto": "", "cabelo": "",
    "roupas": "", "tom_de_pele": "", "genero_aparente": "",
    "idade_aparente": "", "estilo_visual": ""
  }},
  "ambiente": {{
    "tipo": "", "descricao": "", "iluminacao": "",
    "paleta_de_cores": "", "mood": ""
  }},
  "estilo_fotografico": "",
  "sugestoes_de_cena": ["", "", ""],
  "prompt_sd_base": ""
}}"""}],
    max_tokens=1200
)

raw = consolidacao.choices[0].message.content.strip()
if '```' in raw:
    raw = raw.split('```')[1]
    if raw.startswith('json'): raw = raw[4:]
start_idx = raw.find('{')
if start_idx > 0: raw = raw[start_idx:]

perfil_visual = json.loads(raw)

print('\n Perfil visual consolidado:\n')
print(json.dumps(perfil_visual, ensure_ascii=False, indent=2))

output_json_visual = os.path.join(OUTPUT_DIR, f'{nome_musica}_perfil_visual.json')
with open(output_json_visual, 'w', encoding='utf-8') as f:
    json.dump(perfil_visual, f, ensure_ascii=False, indent=2)
print(f'\n Salvo: {output_json_visual}')

##  Célula 7 — Prompt do diretor + Storyboard com LLaMA 3.3

In [ ]:
# ============================================================
#   ESCREVA AQUI — Descreva o clipe como um diretor
# Pressione Enter duas vezes em branco para finalizar
# ============================================================

print(" Descreva o clipe que você quer criar.")
print("   Fale sobre: estilo visual, ambiente, clima, movimentos de câmera,")
print("   o que acontece nas partes intensas e suaves, paleta de cores...")
print("   (quando terminar, deixe uma linha em branco e pressione Enter)\n")

linhas = []
while True:
    linha = input()
    if linha == "" and linhas and linhas[-1] == "":
        break
    linhas.append(linha)

PROMPT_DIRETOR = "\n".join(linhas).strip()

if not PROMPT_DIRETOR:
    raise ValueError(" Prompt vazio! Rode a célula novamente e descreva o clipe.")

print("\n Prompt do diretor recebido:")
print("─" * 50)
print(PROMPT_DIRETOR)
print("─" * 50)

# ============================================================

print(f'\n Gerando storyboard para: {nome_musica}')
print(f'   {len(analise_musical["secoes"])} seções musicais → {len(analise_musical["secoes"])} cenas\n')

prompt = f"""Você é um diretor criativo de clipes musicais. Crie um storyboard cinematográfico.

ANÁLISE MUSICAL:
{json.dumps(analise_musical, ensure_ascii=False)}

PERFIL VISUAL:
{json.dumps(perfil_visual, ensure_ascii=False)}

VISÃO DO DIRETOR:
{PROMPT_DIRETOR}

Crie UMA CENA para cada uma das {len(analise_musical['secoes'])} seções musicais.
Alta intensidade = planos fechados, movimento, energia.
Baixa intensidade = planos abertos, câmera lenta, contemplação.

Retorne APENAS um JSON válido (sem texto extra, sem ```json```):
{{
  "titulo_clipe": "{nome_musica}",
  "conceito_geral": "2 frases descrevendo o conceito",
  "paleta_de_cores": "paleta geral",
  "cenas": [
    {{
      "secao_id": 1,
      "inicio": 0.0,
      "fim": 15.0,
      "duracao": 15.0,
      "intensidade": "baixa",
      "descricao_cena": "descrição cinematográfica detalhada",
      "tipo_de_plano": "plano aberto / close / plano médio / detalhe",
      "movimento_de_camera": "estática / travelling / pan / tilt / drone",
      "mood": "emoção da cena",
      "prompt_sd": "detailed English scene prompt for Stable Diffusion, cinematic, photorealistic",
      "negative_prompt": "blurry, low quality, distorted, bad anatomy",
      "justificativa_musical": "por que esta cena neste momento"
    }}
  ]
}}"""

response = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': prompt}],
    max_tokens=4000,
    temperature=0.7
)

raw = response.choices[0].message.content.strip()
if '```' in raw:
    raw = raw.split('```')[1]
    if raw.startswith('json'): raw = raw[4:]
start_idx = raw.find('{')
if start_idx > 0: raw = raw[start_idx:]

storyboard = json.loads(raw)

print(f" '{storyboard['titulo_clipe']}' — {len(storyboard['cenas'])} cenas geradas\n")
print(f"Conceito : {storyboard['conceito_geral']}")
print(f"Paleta   : {storyboard['paleta_de_cores']}\n")
for cena in storyboard['cenas']:
    print(f"   Cena {cena['secao_id']:2d}  [{cena['inicio']}s → {cena['fim']}s]  "
          f"{cena['intensidade']:5s} | {cena['tipo_de_plano']}")
    print(f"          {cena['mood']} — {cena['descricao_cena'][:75]}...\n")

output_json_sb = os.path.join(OUTPUT_DIR, f'{nome_musica}_storyboard.json')
with open(output_json_sb, 'w', encoding='utf-8') as f:
    json.dump(storyboard, f, ensure_ascii=False, indent=2)
print(f' Salvo: {output_json_sb}')

##  Célula 8 — Visualização do Storyboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

n = len(storyboard['cenas'])
fig, ax = plt.subplots(figsize=(16, max(6, n * 1.4)))
ax.set_xlim(0, analise_musical['duracao_total'])
ax.set_ylim(0, n)
ax.set_xlabel('Tempo (segundos)', fontsize=11)
ax.set_title(
    f"Storyboard — {storyboard['titulo_clipe']}\n{storyboard['conceito_geral'][:110]}",
    fontsize=12, pad=14
)
ax.set_yticks([])

cor_int = {'baixa': '#90CAF9', 'média': '#FFB74D', 'alta': '#EF5350'}
for i, cena in enumerate(storyboard['cenas']):
    y_pos = n - i - 1
    cor   = cor_int.get(cena['intensidade'], '#BDBDBD')
    dur   = cena['fim'] - cena['inicio']
    rect  = mpatches.FancyBboxPatch(
        (cena['inicio'] + 0.3, y_pos + 0.08), max(dur - 0.5, 1), 0.84,
        boxstyle='round,pad=0.02', facecolor=cor, alpha=0.85,
        edgecolor='white', linewidth=1.2
    )
    ax.add_patch(rect)
    ax.text(
        cena['inicio'] + dur / 2, y_pos + 0.5,
        f"#{cena['secao_id']} {cena['tipo_de_plano']}\n{cena['mood'][:32]}",
        ha='center', va='center', fontsize=7.5, fontweight='bold', color='#1a1a1a'
    )
    ax.text(cena['inicio'] + 0.5, y_pos + 0.12,
            f"{cena['inicio']:.0f}s", fontsize=7, color='#444')

ax.legend(handles=[
    mpatches.Patch(color='#90CAF9', label='Baixa intensidade'),
    mpatches.Patch(color='#FFB74D', label='Média intensidade'),
    mpatches.Patch(color='#EF5350', label='Alta intensidade'),
], loc='lower right', fontsize=9)

for t in range(0, int(analise_musical['duracao_total']), 10):
    ax.axvline(t, color='#ccc', linewidth=0.5, zorder=0)

plt.tight_layout()
output_sb_img = os.path.join(OUTPUT_DIR, f'{nome_musica}_storyboard.png')
plt.savefig(output_sb_img, dpi=150, bbox_inches='tight')
plt.show()
print(f' Salvo: {output_sb_img}')

##  Célula 9 — Download dos resultados

In [ ]:
import zipfile
from google.colab import files

# Nome do zip usa o nome da música para não misturar projetos
zip_path = os.path.join(BASE_DIR, f'{nome_musica}_fase1.zip')

arquivos_para_zipar = [
    output_json_musical,
    output_json_visual,
    output_json_sb,
    output_grafico,
    output_sb_img,
]

print(f' Empacotando resultados de "{nome_musica}"...')
with zipfile.ZipFile(zip_path, 'w') as z:
    for arq in arquivos_para_zipar:
        if arq and os.path.exists(arq):
            z.write(arq, os.path.basename(arq))
            print(f'   ✓ {os.path.basename(arq)}')
        else:
            print(f'  X  Não encontrado: {arq}')

print(f'\n Baixando {os.path.basename(zip_path)}...')
files.download(zip_path)

print('\n Fase 1 concluída!')
print('═' * 50)
print(f'  Música    : {nome_musica}')
print(f'  BPM       : {analise_musical["bpm"]}')
print(f'  Duração   : {analise_musical["duracao_total"]}s  ({analise_musical["duracao_total"]/60:.1f} min)')
print(f'  Seções    : {len(analise_musical["secoes"])}')
print(f'  Imagens   : {len(IMAGE_PATHS)}')
print(f'  Cenas     : {len(storyboard["cenas"])}')
print('═' * 50)
print('\n Para fazer outro clipe: volte à Célula 3 e faça upload de outra música + imagens.')
print('  Próxima etapa: Fase 2 — Geração de frames com Stable Diffusion + IP-Adapter')

---
##  Arquivos gerados

Todos os arquivos usam o **nome da sua música** como prefixo, ex: `minhamusica_storyboard.json`.

| Arquivo | Conteúdo | Usado em |
|---|---|---|
| `*_analise.json` | BPM, seções, caminhos Demucs | Fase 2 e 3 |
| `*_perfil_visual.json` | Perfil do personagem e ambiente | Fase 2 |
| `*_storyboard.json` | Cenas com prompts para SD | Fase 2 |
| `*_analise.png` | Gráfico waveform + energia | Referência |
| `*_storyboard.png` | Timeline colorida das cenas | Referência |

##  Para usar com outra música
1. Volte para a **Célula 3** e rode novamente
2. Faça upload da nova música e novas imagens
3. Rode as células 4 a 9 normalmente
4. Os arquivos antigos **não serão sobrescritos** — cada música gera arquivos com seu próprio nome